## 1. Leave-One-Out (LOO)

**Leave-One-Out** — это частный случай кросс-валидации, при котором количество фолдов равно числу объектов в выборке:  
$F = N$, где $N$ — размер выборки. На каждой итерации $i$ модель обучается на всех объектах, кроме $i$-го, и тестируется на этом одном исключённом объекте. Итоговая ошибка усредняется по всем $N$ итерациям:

$$
\text{Err}_{\text{LOO}} = \frac{1}{N} \sum_{i=1}^{N} L\big(y_i, \hat{f}_{-i}(x_i)\big)
$$

где $\hat{f}_{-i}$ — модель, обученная без $i$-го объекта, а $L$ — функция потерь.

**Преимущества (strengths):**
- Минимальное смещение (bias) оценки качества.
- Детерминированность (в отличие от случайной разбивки на K folds).
- Полезна при очень малых выборках.

**Ограничения (limitations):**
- Высокая вычислительная стоимость: $O(N \cdot T)$, где $T$ — время обучения одной модели.
- Высокая дисперсия оценки.
- Для нестабильных моделей может переоценивать ошибку на выбросах.


## 2. Grid Search, Randomized Grid Search и Bayesian optimization

### Grid Search (сеточный перебор)

Задаётся сетка гиперпараметров. Пусть параметры:  
$\theta_1 \in \{\theta_1^{(1)}, \dots, \theta_1^{(k_1)}\},\; \dots,\; \theta_d \in \{\theta_d^{(1)}, \dots, \theta_d^{(k_d)}\}$.  
Общее число комбинаций:

$$
M = \prod_{j=1}^{d} k_j
$$

Модель обучается и оценивается на каждой комбинации. Выбирается комбинация, дающая наилучшее качество на валидации.

**Strengths:** простота, гарантированный поиск по сетке, легко параллелится.  
**Limitations:** проклятие размерности ($M$ растёт экспоненциально), неэффективен для непрерывных параметров.


### Randomized Grid Search (случайный поиск)

Вместо фиксированной сетки задаются распределения для каждого гиперпараметра:  
$\theta_j \sim P_j$. Выбирается случайное число комбинаций $M_{\text{rand}}$ (например, 100).  
Ожидаемое качество поиска:

$$
P\approx 1 - (1 - 0.05)^{M_{\text{rand}}}
$$

**Strengths:** эффективнее сеточного при высокой размерности, может найти хорошую область при редкой сетке.  
**Limitations:** нет гарантии попадания в оптимум.


### Bayesian optimization

Строится суррогатная модель (обычно гауссовский процесс) для целевой функции $f(\theta)$ — ошибки на валидации.  
Гауссовский процесс задаётся средним $\mu(\theta)$ и ковариацией (ядром) $k(\theta, \theta')$, например, RBF-ядро:

$$
k(\theta, \theta') = \exp\left(-\frac{\|\theta - \theta'\|^2}{2\ell^2}\right)
$$

Апостериорное распределение:  
$f(\theta) \sim \mathcal{GP}\big(\mu(\theta),\, k(\theta, \theta')\big)$.

Acquisition function (например, Expected Improvement):

$$
\text{EI}(\theta) = \mathbb{E}\big[\max(0,\, f(\theta_{\text{best}}) - f(\theta))\big]
$$

Следующая точка выбирается максимизацией EI. Цикл повторяется $T$ раз (обычно 50–200).

**Strengths:** требует мало итераций, работает с любыми гиперпараметрами.  
**Limitations:** сложность реализации, зависимость от ядра, менее эффективен при размерности $>20$.


## 3. Классификация методов отбора признаков

| Тип | Описание | Примеры |
|------|-----------|----------|
| **Filter** | Оценивают признаки независимо от модели | Pearson, Chi2, mutual information, mRMR |
| **Wrapper** | Используют модель для оценки подмножеств | RFE, forward/backward selection |
| **Embedded** | Отбор встроен в процесс обучения | Lasso, Ridge, деревья решений |


## 4. Корреляция Пирсона и критерий хи-квадрат 

### Корреляция Пирсона

Для двух непрерывных переменных $X$ и $Y$ выборочный коэффициент корреляции:

$$
r_{XY} = \frac{\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^{n} (x_i - \bar{x})^2} \cdot \sqrt{\sum_{i=1}^{n} (y_i - \bar{y})^2}} = \frac{\text{Cov}(X,Y)}{\sigma_X \sigma_Y}
$$

- $r \in [-1, 1]$
- $|r|$ близок к 1 → сильная линейная связь

**Применение в отборе признаков:** Ранжирование признаков по $|r|$ с целевой переменной.  
**Ограничение:** Не обнаруживает нелинейные зависимости (например, $Y = X^2$ даст $r \approx 0$).


### Критерий хи-квадрат (Chi2)

Для таблицы сопряжённости $r \times c$ с наблюдаемыми частотами $O_{ij}$ и ожидаемыми $E_{ij}$:

$$
\chi^2 = \sum_{i=1}^{r} \sum_{j=1}^{c} \frac{(O_{ij} - E_{ij})^2}{E_{ij}}
$$

Ожидаемая частота при независимости:

$$
E_{ij} = \frac{(\text{сумма по строке } i) \cdot (\text{сумма по столбцу } j)}{\text{общая сумма}}
$$

Распределение $\chi^2$ имеет $(r-1)(c-1)$ степеней свободы. Большое значение → признак зависим от целевой переменной.

**Применение:** Для категориальных признаков и категориальной целевой переменной.  
**Ограничение:** Чувствителен к малым ожидаемым частотам ($E_{ij} < 5$).


## 5. Lasso (L1-регуляризация)

**Целевая функция** для линейной регрессии с матрицей признаков $A \in \mathbb{R}^{m \times n}$, вектором откликов $y \in \mathbb{R}^m$ и вектором коэффициентов $x \in \mathbb{R}^n$:

$$
\hat{x} = \arg\min_{x} \left\{ \|A x - y\|_2^2 + \alpha \|x\|_1 \right\}
$$

где  

- $\|A x - y\|_2^2 = \sum_{i=1}^{m} \big((A x)_i - y_i\big)^2$ — среднеквадратичная ошибка,  
- $\|x\|_1 = \sum_{j=1}^{n} |x_j|$ — L1-норма (штраф),  
- $\alpha \ge 0$ — гиперпараметр регуляризации.

**Почему обнуляет коэффициенты:**  
Задача эквивалентна минимизации $\|A x - y\|_2^2$ при ограничении $\|x\|_1 \le t$. L1-шар имеет углы на осях координат, поэтому оптимальное решение часто достигается в таких углах, где некоторые $x_j = 0$.

**Условие субградиента:**  

$$
-2A^T(y - A x) + \alpha \cdot \partial \|x\|_1 \ni 0
$$

где $\partial \|x\|_1$ — субдифференциал:

$$
(\partial \|x\|_1)_j = 
\begin{cases} 
\operatorname{sign}(x_j), & x_j \neq 0 \\ 
[-1, 1], & x_j = 0 
\end{cases}
$$

**Strengths:** автоматический отбор признаков, устойчивость к избыточности.  
**Limitations:** при $n > m$ выбирает не более $m$ признаков, нестабилен, из группы коррелированных признаков оставляет только один.

> **Сравнение с Ridge (L2):**  
> Ridge решает $\min \|A x - y\|_2^2 + \alpha \|x\|_2^2$, где $\|x\|_2^2 = \sum_{j=1}^{n} x_j^2$. Он сжимает коэффициенты, но **не обнуляет** их, поэтому Ridge **не является** методом отбора признаков.


## 6. Permutation significance (значимость перестановками)

**Алгоритм:**  
Пусть обучена модель $f$, есть валидационная выборка $\{(x_i, y_i)\}_{i=1}^{M}$ и метрика качества $Q$ (например, accuracy).

1. Вычисляем baseline-качество: $Q_{\text{base}} = Q(f, X_{\text{val}}, y_{\text{val}})$.
2. Для каждого признака $j$:
   - Создаём $X_{\text{val}}^{(j)}$ — копию $X_{\text{val}}$ с перемешанными значениями признака $j$.
   - Вычисляем $Q_{\text{perm}}^{(j)} = Q(f, X_{\text{val}}^{(j)}, y_{\text{val}})$.
   - Важность признака $j$: $\text{Importance}_j = Q_{\text{base}} - Q_{\text{perm}}^{(j)}$.
3. Повторяем шаг 2 несколько раз (например, $R = 10$) и усредняем.

**Формула (средняя важность после $R$ повторений):**

$$
\text{Importance}_j = \frac{1}{R} \sum_{r=1}^{R} \left( Q_{\text{base}} - Q_{\text{perm}}^{(j,r)} \right)
$$

**Strengths:** модельно-независимый, интерпретируемый результат (падение метрики).  
**Limitations:** вычислительно дорог ($O(R \cdot n \cdot M \cdot T_{\text{pred}})$), корреляции между признаками искажают оценку, не для всех метрик корректен.


## 7. SHAP 

**Основная формула (аддитивность):**  
Для объекта $x$ предсказание модели $f(x)$ раскладывается на сумму:

$$
f(x) = \phi_0 + \sum_{j=1}^{M} \phi_j
$$

где  

- $\phi_0 = \mathbb{E}[f(X)]$ — среднее предсказание модели (baseline),  
- $\phi_j$ — вклад $j$-го признака в предсказание для данного $x$.

**Значение Шепли (теоретико-игровая формула):**

$$
\phi_j = \sum_{S \subseteq \{1,\dots,M\} \setminus \{j\}} \frac{|S|! \, (M - |S| - 1)!}{M!} \Big[ f_{x}(S \cup \{j\}) - f_{x}(S) \Big]
$$

где $f_{x}(S)$ — предсказание модели при условии, что известны признаки из множества $S$ (а остальные маргинализованы либо усреднены по распределению).

**На практике используют аппроксимации:**  
- **TreeSHAP** (для деревьев решений) — точное вычисление за $O(M \cdot \text{глубина})$.  
- **KernelSHAP** (для любых моделей) — аппроксимация через взвешенную линейную регрессию.

**Аналитическая форма KernelSHAP:**  
Минимизируется взвешенная MSE:

$$
\min_{\phi_0,\dots,\phi_M} \sum_{z \in \{0,1\}^M} w(z) \left( f_x(z) - \big(\phi_0 + \sum_{j=1}^M \phi_j z_j\big) \right)^2
$$

где веса  

$$
w(z) = \frac{M-1}{\binom{M}{|z|} \cdot |z| \cdot (M-|z|)}
$$

а $f_x(z)$ — предсказание модели при подстановке $z$ (признак включён, если $z_j=1$).

**Strengths:**  
- Единая теоретическая основа (аксиомы Шепли).  
- Свойство согласованности.  
- Локальное (для одного объекта) и глобальное (усреднение) объяснение.

**Limitations:**  
- Высокая вычислительная сложность: KernelSHAP требует $O(2^M)$ в точной версии.  
- Неоднозначность при сильной корреляции признаков.  
- SHAP даёт разложение предсказания, а не причинно-следственную связь.

In [248]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.linear_model import Lasso, ElasticNet
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, GroupKFold, TimeSeriesSplit
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score, mean_absolute_percentage_error
from itertools import product
from sklearn.inspection import permutation_importance
from sklearn.base import clone
import optuna
import shap
import time

In [249]:
train_df = pd.read_json("train.json")
test_df = pd.read_json("test.json")
train_df.head(3)

,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address,interest_level
4,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[Dining Room, Pre-War, Laundry in Building, Di...",40.7108,7170325,-73.9539,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/7170325_3bb5ac84...,2400,145 Borinquen Place,medium
6,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...,East 44th,"[Doorman, Elevator, Laundry in Building, Dishw...",40.7513,7092344,-73.9722,955db33477af4f40004820b4aed804a0,[https://photos.renthop.com/2/7092344_7663c19a...,3800,230 East 44th,low
9,1.0,2,cd759a988b8f23924b5a2058d5ab2b49,2016-06-14 15:19:59,**FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...,East 56th Street,"[Doorman, Elevator, Laundry in Building, Laund...",40.7575,7158677,-73.9625,c8b10a317b766204f08e613cef4ce7a0,[https://photos.renthop.com/2/7158677_c897a134...,3495,405 East 56th Street,medium


In [250]:
from collections import Counter

class FeatureExtractor:
    def __init__(self, n=20):
        self.n = n
        self.top_features = None
        
    def clean_features(self, features: list) -> list:
        if not isinstance(features, list):
            return []
        return [feature.replace(' ', '') for feature in features]
    
    def fit(self, df):
        df['clean_features'] = df['features'].apply(self.clean_features)
        
        all_features = []
        for feature in df['clean_features']:
            all_features.extend(feature)
            
        counter = Counter(all_features)
        self.top_features = [name for name, _ in counter.most_common(self.n)]
        
        print(f"Всего признаков: {len(all_features)}")
        print(f"Уникальных: {len(counter)}")
        print(f"Топ-{self.n}: {self.top_features[:5]}...") 
        
        return self
    
    def transform(self, df):
        df = df.copy()
        
        if 'clean_features' not in df.columns:
            df['clean_features'] = df['features'].apply(self.clean_features)
            
        for feature in self.top_features:
            df[feature] = df['clean_features'].apply(
                lambda x: 1 if feature in x else 0
            )
            
        feature_cols = ['bathrooms', 'bedrooms', 'created']
        feature_cols += [col for col in df.columns if col in self.top_features]  
        df.drop('clean_features', axis=1, inplace=True)
        
        return df[feature_cols]
    
    def fit_transform(self, df):
        self.fit(df)
        return self.transform(df)

In [251]:
extractor = FeatureExtractor(n=20)
X = extractor.fit_transform(train_df)
y = train_df['price']
X.head(3)

Всего признаков: 267906
Уникальных: 1548
Топ-20: ['Elevator', 'CatsAllowed', 'HardwoodFloors', 'DogsAllowed', 'Doorman']...


,bathrooms,bedrooms,created,Elevator,CatsAllowed,HardwoodFloors,DogsAllowed,Doorman,Dishwasher,NoFee,...,LaundryinUnit,RoofDeck,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace
4,1.0,1,2016-06-16 05:55:27,0,1,1,1,0,1,0,...,0,0,0,1,0,0,0,0,0,0
6,1.0,2,2016-06-01 05:44:33,1,0,1,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
9,1.0,2,2016-06-14 15:19:59,1,0,1,0,1,1,0,...,1,0,0,0,0,0,0,0,0,0


In [252]:
def split_data(data, train_idx, val_idx=None, test_idx=None):
   
    if isinstance(data, pd.DataFrame):
        train_data = data.iloc[train_idx]
        test_data = data.iloc[test_idx]
        if val_idx is not None:
            val_data = data.iloc[val_idx]
            return train_data, val_data, test_data
        return train_data, test_data
    else:
        train_data = data[train_idx]
        test_data = data[test_idx]
        if val_idx is not None:
            val_data = data[val_idx]
            return train_data, val_data, test_data
        return train_data, test_data

In [253]:
def my_train_test_split(X, y=None, test_size=0.2, random_state=None):
    
    X = X.reset_index(drop=True)
    if y is not None:
        y = y.reset_index(drop=True)
        
    if random_state is not None:
        np.random.seed(random_state)
        
    n_samples = len(X)
    
    indices = np.arange(n_samples)
    np.random.shuffle(indices)
    
    if 0 < test_size < 1:
        test_end = int(n_samples * test_size)
    else:
        ValueError('Параметр test_size принимает значения в диапазоне (0, 1)')
        
    test_indices = indices[:test_end]
    train_indices = indices[test_end:]
    
    if y is not None:
        if len(y) != len(X):
            raise ValueError(f"X и y должны иметь длину: {n_samples} vs {len(y)}")
        X_train, X_test = split_data(X, train_indices, test_idx=test_indices)
        y_train, y_test = split_data(y, train_indices, test_idx=test_indices)
        return X_train, X_test, y_train, y_test
    else:
        return split_data(X, train_indices, test_indices)

In [254]:
def my_train_val_test_split(X, y=None, val_size=0.15, test_size=0.15, random_state=None):
    
    X = X.reset_index(drop=True)
    if y is not None:
        y = y.reset_index(drop=True)
        
    if random_state is not None:
        np.random.seed(random_state)
        
    n_samples = len(X)
    
    indices = np.arange(n_samples)
    np.random.shuffle(indices)
    
    if test_size + val_size >= 1:
        ValueError('Сумма тестовой и валидационной части должна быть меньше 1')
        
    test_end = int(n_samples * test_size)
    val_end = test_end + int(n_samples * val_size)
    
    test_indices = indices[:test_end]
    val_indices = indices[test_end:val_end]
    train_indices = indices[val_end:]
    
    if y is not None:
        if len(y) != len(X):
            raise ValueError(f"X и y должны иметь длину: {n_samples} vs {len(y)}")
        else: 
            X_train, X_val, X_test = split_data(X, train_indices, val_indices, test_indices)
            y_train, y_val, y_test = split_data(y, train_indices, val_indices, test_indices)
        return X_train, X_val, X_test, y_train, y_val, y_test
    else:
        return split_data(X, train_indices, val_indices, test_indices)
    

In [255]:
def date_split_2(df, date_split, target_col, date_column='created'):
    df = df.copy()
    df[date_column] = pd.to_datetime(df[date_column])
    split_date = pd.to_datetime(date_split)
    
    train_df = df[df[date_column] < split_date].copy()
    test_df = df[df[date_column] >= split_date].copy()
    
    if len(train_df) == 0:
        raise ValueError(f"Неверное значение для тренировочной выборки, date_split={date_split}")
    if len(test_df) == 0:
        raise ValueError(f"Неверное значение для тестовой выборки,  date_split={date_split}")
    
    X_train = train_df.drop(target_col, axis=1)
    y_train = train_df[target_col]
    X_test = test_df.drop(target_col, axis=1)
    y_test = test_df[target_col]
    
    return X_train, X_test, y_train, y_test

In [256]:
def date_split_3(X, y, date_column, val_size=0.2, test_size=0.2):
    df = X.copy()
    df[date_column] = pd.to_datetime(df[date_column])
    df['_target'] = y
    df_sorted = df.sort_values(date_column).reset_index(drop=True)
    
    n_samples = len(df_sorted)
    test_end = n_samples
    test_start = int(n_samples * (1 - test_size))
    val_end = test_start
    val_start = int(n_samples * (1 - val_size - test_size))
    
    if val_start <= 0:
        raise ValueError(f"val_size + test_size = {val_size + test_size} слишком большие")
    
    train_df = df_sorted.iloc[:val_start].copy()
    val_df = df_sorted.iloc[val_start:val_end].copy()
    test_df = df_sorted.iloc[test_start:test_end].copy()
    
    X_train = train_df.drop(columns=[date_column, '_target'])
    X_val = val_df.drop(columns=[date_column, '_target'])
    X_test = test_df.drop(columns=[date_column, '_target'])
    
    y_train = train_df['_target']
    y_val = val_df['_target']
    y_test = test_df['_target']
    
    return X_train, X_val, X_test, y_train, y_val, y_test

Детерминированное разделение — это свойство алгоритма или функции, при котором при многократном запуске с одними и теми же входными данными результат всегда будет одинаковым.

In [257]:
def k_fold(df, k, random_state=21):
    
    np.random.seed(random_state)
    indices = np.arange(len(df))
    np.random.shuffle(indices)
    folds = np.array_split(indices, k)
    
    res = []
    for i in range(k):
        test_indices = folds[i].tolist()
        train_indices = np.concatenate([folds[j] for j in range(k) if j != i])
        res.append([train_indices, test_indices])
        
    return res

In [258]:
folds = k_fold(X, k=3)

for i, (train_idx, test_idx) in enumerate(folds):
    print(f"Fold {i+1}:")
    print(f"Train indices: {train_idx}")
    print(f"Test indices: {test_idx}")
    print(f"Train data: {X.iloc[train_idx]}")
    print(f"Test data: {X.iloc[test_idx]}")

Fold 1:
Train indices: [ 1606 27082 14571 ...  5944  5327 15305]
Test indices: [10451, 9985, 13712, 35989, 21892, 25920, 44367, 4052, 46022, 31803, 46473, 36534, 25926, 11486, 37374, 9455, 48281, 15103, 37167, 30691, 5568, 14496, 9953, 29631, 16135, 18587, 42644, 13013, 30601, 27402, 34553, 24458, 1248, 6364, 40115, 161, 29499, 26795, 25735, 42851, 42362, 43895, 3276, 6879, 36858, 32500, 12194, 19562, 31462, 48748, 38339, 48552, 40067, 33830, 3278, 46285, 45057, 47713, 18455, 22413, 5970, 37116, 9083, 24788, 34170, 34868, 19126, 36956, 37408, 28925, 33605, 8480, 38229, 24575, 25490, 3531, 10954, 35240, 47581, 11977, 45207, 29002, 8266, 3717, 38266, 40678, 1575, 34811, 14331, 12818, 10658, 15023, 21526, 7245, 3806, 13913, 42961, 6246, 18436, 28700, 40717, 2633, 45319, 30150, 23989, 14773, 38507, 30402, 34036, 12523, 7964, 33127, 4066, 1211, 9180, 28233, 3028, 4421, 36847, 23026, 16081, 15825, 255, 7056, 35370, 22782, 151, 11238, 3351, 5049, 21899, 13415, 37187, 1098, 34594, 23321, 24504

In [259]:
def group_k_fold(df, k, group_field, random_state=21):
    
    df = df.reset_index(drop=True)
    
    np.random.seed(random_state)
    unique_groups = df[group_field].unique()
    np.random.shuffle(unique_groups)
    folds = np.array_split(unique_groups, k)
    
    res = []
    for i in range(k):
        test_groups = set(folds[i])
        test_indices = df[df[group_field].isin(test_groups)].index.to_list()
        train_indices = df[~df[group_field].isin(test_groups)].index.to_list()
        res.append([train_indices, test_indices])
        
    return res

In [260]:
folds = group_k_fold(X, k=3, group_field='bedrooms')

for i, (train_idx, test_idx) in enumerate(folds):
    print(f"Fold {i+1}:")
    print(f"Train groups: {np.unique(X['bedrooms'].iloc[train_idx])}")
    print(f"Test groups: {np.unique(X['bedrooms'].iloc[test_idx])}")
    print(f"Train size: {len(train_idx)}, Test size: {len(test_idx)}")
    print()

Fold 1:
Train groups: [0 1 4 5 6 7]
Test groups: [2 3 8]
Train size: 27451, Test size: 21901

Fold 2:
Train groups: [1 2 3 4 7 8]
Test groups: [0 5 6]
Train size: 39584, Test size: 9768

Fold 3:
Train groups: [0 2 3 5 6 8]
Test groups: [1 4 7]
Train size: 31669, Test size: 17683



In [261]:
def stratified_kfold(df, k, stratify_field, random_state=21):
    
    df = df.copy()
    df['_original_index'] = df.index
    unique_classes = df[stratify_field].unique()
    class_indices = {}
    
    for cls in unique_classes:
        class_indices[cls] = df[df[stratify_field] == cls]['_original_index'].to_list()
        
    for cls in class_indices:
        np.random.shuffle(class_indices[cls])
    
    folds = [[] for _ in range(k)]
    
    res = []
    for cls in unique_classes:
        cls_idx_list = class_indices[cls]
        for i, idx in enumerate(cls_idx_list):
            fold_idx = i % k
            folds[fold_idx].append(idx)
    
    res = []
    for i in range(k):
        test_indices = folds[i]
        train_indices = []
        for j in range(k):
            if j != i:
                train_indices.extend(folds[j])
        res.append((train_indices, test_indices))
    
    return res

In [262]:
folds = stratified_kfold(X, k=3, stratify_field='bathrooms', random_state=42)

for i, (train_idx, test_idx) in enumerate(folds):
    print(f"Fold {i+1}: train={len(train_idx)}, test={len(test_idx)}")

    train_bathrooms = X.loc[train_idx]['bathrooms'].value_counts(normalize=True)
    test_bathrooms = X.loc[test_idx]['bathrooms'].value_counts(normalize=True)
    
    print(f"  Train:")
    for bathrooms, prop in train_bathrooms.items():
        print(f"    {bathrooms}: {prop:.2%}")
    print(f"  Test:")
    for bathrooms, prop in test_bathrooms.items():
        print(f"    {bathrooms}: {prop:.2%}")
    print()

Fold 1: train=32894, test=16458
  Train:
    1.0: 79.90%
    2.0: 15.52%
    3.0: 1.51%
    1.5: 1.31%
    0.0: 0.63%
    2.5: 0.56%
    4.0: 0.32%
    3.5: 0.14%
    4.5: 0.06%
    5.0: 0.04%
    5.5: 0.01%
    6.0: 0.01%
  Test:
    1.0: 79.85%
    2.0: 15.52%
    3.0: 1.51%
    1.5: 1.31%
    0.0: 0.64%
    2.5: 0.57%
    4.0: 0.32%
    3.5: 0.15%
    4.5: 0.06%
    5.0: 0.04%
    6.0: 0.01%
    5.5: 0.01%
    6.5: 0.01%
    7.0: 0.01%
    10.0: 0.01%

Fold 2: train=32903, test=16449
  Train:
    1.0: 79.87%
    2.0: 15.52%
    3.0: 1.51%
    1.5: 1.31%
    0.0: 0.64%
    2.5: 0.56%
    4.0: 0.32%
    3.5: 0.14%
    4.5: 0.06%
    5.0: 0.04%
    6.0: 0.01%
    5.5: 0.01%
    6.5: 0.00%
    7.0: 0.00%
    10.0: 0.00%
  Test:
    1.0: 79.89%
    2.0: 15.52%
    3.0: 1.51%
    1.5: 1.31%
    0.0: 0.63%
    2.5: 0.56%
    4.0: 0.32%
    3.5: 0.14%
    4.5: 0.06%
    5.0: 0.04%
    5.5: 0.01%
    6.0: 0.01%

Fold 3: train=32907, test=16445
  Train:
    1.0: 79.87%
    2.0: 15.52%
    3.0

In [263]:
def time_split(df, k, date_field):
    
    df = df.copy()
    df[date_field] = pd.to_datetime(df[date_field])
    df_sorted = df.sort_values(date_field).reset_index(drop=True)
    
    n_samples = len(df)
    fold_size = n_samples // k
    
    res = []
    for i in range(k):
        test_end = n_samples - (k - 1 - i) * fold_size
        test_start = test_end - fold_size
        
        if i == k - 1:
            test_end = n_samples
            
        test_indices = df_sorted.index[test_start:test_end].tolist()
        train_indices = df_sorted.index[:test_start].tolist()
        
        if len(train_indices) > 0 and len(test_indices) > 0:
            res.append((train_indices, test_indices))

    return res

In [264]:
folds = time_split(X, k=4, date_field='created')

print(f"\nВсего объектов: {len(X)}")
print(f"Размер фолда: ~{len(X) // 4} объектов")
print(f"Создано фолдов: {len(folds)}\n")

for i, (train_idx, test_idx) in enumerate(folds):
    train_dates = X.iloc[train_idx]['created']
    test_dates = X.iloc[test_idx]['created']
    
    print(f"Fold {i+1}:")
    print(f"  Train: {len(train_idx)} объектов")
    print(f"  Test:  {len(test_idx)} объектов")
  


Всего объектов: 49352
Размер фолда: ~12338 объектов
Создано фолдов: 3

Fold 1:
  Train: 12338 объектов
  Test:  12338 объектов
Fold 2:
  Train: 24676 объектов
  Test:  12338 объектов
Fold 3:
  Train: 37014 объектов
  Test:  12338 объектов


In [265]:
def compare_cross_validation(X, y, groups, dates, k=5):
    res = {}
    
    print('K-Fold')
    my_fold = k_fold(X, k, random_state=21)
    sklearn_kfold = KFold(n_splits=k, shuffle=True, random_state=21)
    sklearn_fold = list(sklearn_kfold.split(X))
    
    print(f"Количество фолдов: {k}")
    print(f"My realization фолды: {len(my_fold)}")
    print(f"Sklearn фолды: {len(sklearn_fold)}")
    
    my_train, my_test = my_fold[0]
    sklearn_train, sklearn_test = sklearn_fold[0]
    
    print(f"Размер train: my realization={len(my_train)}, sklearn={len(sklearn_train)}")
    print(f"Размер test: my realization={len(my_test)}, sklearn={len(sklearn_test)}")
    
    print(f"Совпадение индексов train: {set(my_train) == set(sklearn_train)}")
    print(f"Совпадение индексов test: {set(my_test) == set(sklearn_test)}\n\n\n")
    
    res['KFold'] = {'my_kfold': my_fold, 'sklearn_kfold': sklearn_fold,
                    'match': set(my_train)==set(sklearn_train) and set(my_test)==set(sklearn_test)}
    
    X_reset = X.reset_index(drop=True)
    y_reset = y.reset_index(drop=True) if hasattr(y, 'reset_index') else pd.Series(y).reset_index(drop=True)
    
    
    
    print("GroupKFold")
    my_group_kfold = group_k_fold(X, k, group_field='bedrooms', random_state=21)
    sklearn_group_kfold = GroupKFold(n_splits=k)
    sklearn_fold = list(sklearn_group_kfold.split(X_reset, y_reset, groups))
    
    my_train, my_test = my_group_kfold[0]
    sklearn_train, sklearn_test = sklearn_fold[0]
    
    print(f"Размер train: my realization={len(my_train)}, sklearn={len(sklearn_train)}")
    print(f"Размер test: my realization={len(my_test)}, sklearn={len(sklearn_test)}")
    
    my_test_groups = set(groups.iloc[my_test])
    sklearn_test_group = set(groups.iloc[sklearn_test])
    
    print(f"Уникальные группы в test (my realization): {my_test_groups}")
    print(f"Уникальные группы в test (sklearn): {sklearn_test_group}\n\n\n")
    
    res['group'] = {'our': my_group_kfold, 'sklearn': sklearn_fold,
                    'match': my_test_groups == sklearn_test_group}
    
    
    
    print('StratifiedKFold')
    X_y = X_reset.copy()
    X_y['y'] = y_reset
    
    my_stratified_kfold = stratified_kfold(X_y, k, stratify_field='y')
    sklearn_stritified_kfold = StratifiedKFold(n_splits=k, shuffle=True)
    sklearn_fold = list(sklearn_stritified_kfold.split(X_reset, y_reset))
    
    my_train, my_test = my_stratified_kfold[0]
    sklearn_train, sklearn_test = sklearn_fold[0]
    
    print(f"Размер train: my realization={len(my_train)}, sklearn={len(sklearn_train)}")
    print(f"Размер test: my realization={len(my_test)}, sklearn={len(sklearn_test)}")
    
    our_y_train = y_reset.iloc[my_train] if hasattr(y_reset, 'iloc') else y_reset[my_train]
    our_y_test = y_reset.iloc[my_test] if hasattr(y_reset, 'iloc') else y_reset[my_test]
    sk_y_train = y_reset.iloc[sklearn_train] if hasattr(y_reset, 'iloc') else y_reset[sklearn_train]
    sk_y_test = y_reset.iloc[sklearn_test] if hasattr(y_reset, 'iloc') else y_reset[sklearn_test]
    
    print(f"\nРаспределение классов в train (my realization): {Counter(our_y_train)}")
    print(f"Распределение классов в train (sklearn): {Counter(sk_y_train)}")
    print(f"Распределение классов в test (my realization: {Counter(our_y_test)}")
    print(f"Распределение классов в test (sklearn): {Counter(sk_y_test)}\n\n\n")
    
    res['stratified'] = {'our': my_stratified_kfold, 'sklearn': sklearn_stritified_kfold,
                             'match': set(my_train) == set(sklearn_train) and set(my_test) == set(sklearn_test)}
    
    
    
    
    print("Time Split")

    X_ts = X_reset.copy()
    if 'created' in X_ts.columns:
        X_ts['created'] = pd.to_datetime(X_ts['created'])
    else:
        X_ts['created'] = pd.to_datetime(dates)

    my_ts = time_split(X_ts, k, 'created')
    sklearn_ts = TimeSeriesSplit(n_splits=k)
    sklearn_fold = list(sklearn_ts.split(X_ts))

    for i in range(min(len(my_ts), len(sklearn_fold))):
        our_train, our_test = my_ts[i]
        sk_train, sk_test = sklearn_fold[i]

        print(f"Фолд: {i+1}")
        print(f"Train размер: my realization={len(our_train)}, sklearn={len(sk_train)}")
        print(f"Test размер: my realization={len(our_test)}, sklearn={len(sk_test)}")

        our_train_dates = pd.to_datetime(dates)[our_train]
        our_test_dates = pd.to_datetime(dates)[our_test]

        print(f"Train dates: {our_train_dates[0]} ... {our_train_dates[-1]}")
        print(f"Test dates: {our_test_dates[0]} ... {our_test_dates[-1]}")

        print(f"Train раньше Test: {max(our_train_dates) < min(our_test_dates)}")

    res['timeseries'] = {'our': my_ts, 'sklearn': sklearn_fold,
                        'match': set(our_train) == set(sk_train) and set(our_test) == set(sk_test)}
    return res

groups = train_df['bedrooms'].copy()
X = X.copy()
y = train_df['price'].copy()
dates = pd.to_datetime(train_df['created']).values
y_disc = pd.cut(pd.Series(y), bins=3, labels=False).astype(int).values
res = compare_cross_validation(X, y_disc, groups, dates, k=3)

K-Fold
Количество фолдов: 3
My realization фолды: 3
Sklearn фолды: 3
Размер train: my realization=32901, sklearn=32901
Размер test: my realization=16451, sklearn=16451
Совпадение индексов train: True
Совпадение индексов test: True



GroupKFold
Размер train: my realization=27451, sklearn=33303
Размер test: my realization=21901, sklearn=16049
Уникальные группы в test (my realization): {8, 2, 3}
Уникальные группы в test (sklearn): {1, 5, 6, 7, 8}



StratifiedKFold
Размер train: my realization=32900, sklearn=32901
Размер test: my realization=16452, sklearn=16451

Распределение классов в train (my realization): Counter({0: 32900})
Распределение классов в train (sklearn): Counter({0: 32900, 2: 1})
Распределение классов в test (my realization: Counter({0: 16451, 2: 1})
Распределение классов в test (sklearn): Counter({0: 16451})



Time Split
Фолд: 1
Train размер: my realization=2, sklearn=12338
Test размер: my realization=16450, sklearn=12338
Train dates: 2016-06-16 05:55:27 ... 2016-06-01 

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(


Train раньше Test: False
Фолд: 3
Train размер: my realization=32902, sklearn=37014
Test размер: my realization=16450, sklearn=12338
Train dates: 2016-06-16 05:55:27 ... 2016-05-13 02:48:59
Test dates: 2016-05-17 05:21:56 ... 2016-04-20 05:34:00
Train раньше Test: False


Лучшая стратегия перекрестной проверки - StritifiedKFold

Так как в целевой переменной классы очень несбалансированны, то лучше всего использовать этот метод, так как он сохраняет пропорцию классов во всех фолдах, благодаря чему не появляется фолдов без малопредставленного класса. Так модель обучается лучше.

In [266]:
X_train, X_val, X_test, y_train, y_val, y_test = date_split_3(
    X,
    y,
    date_column='created',
    val_size=0.2,
    test_size=0.2
)

In [267]:
X_train

,bathrooms,bedrooms,Elevator,CatsAllowed,HardwoodFloors,DogsAllowed,Doorman,Dishwasher,NoFee,LaundryinBuilding,...,LaundryinUnit,RoofDeck,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace
0,1.0,1,1,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1,1.0,0,0,1,0,1,0,0,1,0,...,0,0,0,0,0,0,0,1,0,0
2,2.0,3,1,1,0,1,1,0,1,0,...,0,0,0,0,0,0,0,1,0,0
3,1.0,1,1,1,0,1,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
4,1.0,1,1,1,0,1,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29606,1.0,2,0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
29607,1.0,1,0,1,0,1,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
29608,1.0,2,1,1,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
29609,1.0,0,1,1,0,0,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [268]:
numeric_column = ['bedrooms', 'bathrooms']
scaler = MinMaxScaler()

X_train[numeric_column] = scaler.fit_transform(X_train[numeric_column])
X_val[numeric_column] = scaler.fit_transform(X_val[numeric_column])
X_test[numeric_column] = scaler.fit_transform(X_test[numeric_column])

In [269]:
def calculate_metrics(model, X_train, y_train, X_val, y_val, X_test, y_test, model_name):
    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)
    y_pred_test = model.predict(X_test)
    
    results = []
    for split, y_true, y_pred in [('train', y_train, y_pred_train),
                                   ('val', y_val, y_pred_val),
                                   ('test', y_test, y_pred_test)]:
        results.append({
            'model': model_name,
            'split': split,
            'MAE': mean_absolute_error(y_true, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
            'R2': r2_score(y_true, y_pred)
        })
    return pd.DataFrame(results)


In [270]:
all_results = pd.DataFrame()

In [271]:
alphas = np.logspace(-2, 1, 300)
best_score = -10e-5
best_model = None
best_alpha = None

for a in alphas:
    lasso = Lasso(alpha=a, max_iter=10000, random_state=21)
    lasso.fit(X_train, y_train)
    val_score = lasso.score(X_val, y_val)
    if val_score > best_score:
        best_score = val_score
        best_lasso = lasso
        best_alpha = a
        
y_pred = lasso.predict(X_test)

df_temp = calculate_metrics(best_lasso, X_train, y_train, X_val, y_val, X_test, y_test, 'lasso')
all_results = pd.concat([all_results, df_temp], ignore_index=True)

print(f"\nЛучший alpha: {best_alpha}")
print(f"Test MSE: {mean_squared_error(y_test, y_pred):.4f}")
print(f"Test R2: {r2_score(y_test, y_pred):.4f}\n\n\n")

feature_imp = pd.DataFrame({
    'feature': X_train.columns,
    'coefs': best_lasso.coef_
        })

feature_imp = feature_imp[feature_imp['coefs'] != 0].sort_values(by='coefs', key=abs, ascending=False)
print(f"\nНенулевых коэффициентов: {len(feature_imp)}")
print(f"\nТоп-10 признаков по весам:")
print(feature_imp)


Лучший alpha: 10.0
Test MSE: 2174165169.3959
Test R2: 0.0013




Ненулевых коэффициентов: 16

Топ-10 признаков по весам:
              feature         coefs
0           bathrooms  19347.349302
1            bedrooms   3192.925678
6             Doorman   1087.022262
12      LaundryinUnit    549.570316
9   LaundryinBuilding   -330.097310
19  LaundryInBuilding   -274.721626
8               NoFee   -249.263390
16  HighSpeedInternet   -234.397350
21            Terrace    215.384974
4      HardwoodFloors   -179.154859
5         DogsAllowed    177.596261
15         DiningRoom    141.170363
11            Pre-War    120.627854
2            Elevator    112.207037
13           RoofDeck    -32.881869
10      FitnessCenter    -16.794089


In [272]:
all_results

,model,split,MAE,RMSE,R2
0,lasso,train,947.773239,8979.525492,0.032879
1,lasso,val,1969.391452,2660.601480,0.013286
2,lasso,test,2097.436003,46627.944083,0.001324


In [273]:
top10_lasso = feature_imp.head(10)['feature'].to_list()
X_train_lasso = X_train[top10_lasso]
X_val_lasso = X_val[top10_lasso]
X_test_lasso = X_test[top10_lasso]

start_time = time.time()
lasso_top10 = Lasso(alpha=best_alpha, max_iter=10000, random_state=21)
lasso_top10.fit(X_train_lasso, y_train)
lasso_time = time.time() - start_time
print(f"Время обучения: {lasso_time:.2f} сек")
df_temp = calculate_metrics(lasso_top10, X_train_lasso, y_train, X_val_lasso, y_val, X_test_lasso, y_test, 'lasso_top10')
all_results = pd.concat([all_results, df_temp], ignore_index=True)
print(df_temp)

Время обучения: 0.03 сек
         model  split          MAE          RMSE        R2
0  lasso_top10  train   948.297320   8980.694522  0.032627
1  lasso_top10    val  1977.672451   2668.067386  0.007740
2  lasso_top10   test  2106.725617  46629.438912  0.001260


In [274]:
correlations = []
for col in X_train.columns:
    if X_train[col].isnull().sum() / len(X_train) < 0.5:
        corr = abs(X_train[col].corr(y_train))
        correlations.append((col, corr))
    
correlations.sort(key=lambda x: x[1], reverse=True)
selected_corr = [col for col, _ in correlations[:10]]
corr_time = time.time() - start_time

print(f"Время отбора: {corr_time:.4f} сек")
print(f"Отобранные признаки: {selected_corr}\n\n\n")
print(pd.DataFrame(correlations))

Время отбора: 0.5316 сек
Отобранные признаки: ['bathrooms', 'bedrooms', 'Doorman', 'LaundryinUnit', 'DiningRoom', 'Elevator', 'FitnessCenter', 'Dishwasher', 'Terrace', 'OutdoorSpace']



                    0         1
0           bathrooms  0.168800
1            bedrooms  0.117016
2             Doorman  0.068461
3       LaundryinUnit  0.058185
4          DiningRoom  0.051676
5            Elevator  0.043386
6       FitnessCenter  0.041127
7          Dishwasher  0.038111
8             Terrace  0.037136
9        OutdoorSpace  0.031051
10            Balcony  0.026211
11       SwimmingPool  0.025064
12           RoofDeck  0.023317
13        DogsAllowed  0.022517
14  LaundryinBuilding  0.020913
15        CatsAllowed  0.019390
16              NoFee  0.018702
17  HighSpeedInternet  0.016809
18    NewConstruction  0.012469
19     HardwoodFloors  0.011983
20  LaundryInBuilding  0.006337
21            Pre-War  0.003823


In [275]:
X_train_corr = X_train[selected_corr]
X_val_corr = X_val[selected_corr]
X_test_corr = X_test[selected_corr]

start_time = time.time()
lasso_corr = Lasso(alpha=best_alpha, max_iter=10000, random_state=21)
lasso_corr.fit(X_train_corr, y_train)
corr_time = time.time() - start_time
print(f"Время отбора: {corr_time:.4f} сек")

df_temp = calculate_metrics(lasso_corr, X_train_corr, y_train, X_val_corr, y_val, X_test_corr, y_test, 'lasso_corr')
all_results = pd.concat([all_results, df_temp], ignore_index=True)
print(df_temp)

Время отбора: 0.0141 сек
        model  split          MAE          RMSE        R2
0  lasso_corr  train   952.037483   8984.689171  0.031766
1  lasso_corr    val  1994.974778   2684.681828 -0.004656
2  lasso_corr   test  2127.033282  46633.164966  0.001100


In [276]:
perm_imp = permutation_importance(best_lasso, X_val, y_val, n_repeats=10, random_state=21)

perm_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': perm_imp.importances_mean
})

top10_perm = perm_df.head(10)['feature'].tolist()
perm_df

,feature,importance
0,bathrooms,0.589924
1,bedrooms,0.046689
2,Elevator,0.000349
3,CatsAllowed,0.000000
4,HardwoodFloors,0.003256
5,DogsAllowed,0.000390
6,Doorman,0.060518
7,Dishwasher,0.000000
8,NoFee,0.007412
9,LaundryinBuilding,0.008614


In [277]:
X_train_perm = X_train[top10_perm]
X_val_perm = X_val[top10_perm]
X_test_perm = X_test[top10_perm]

start_time = time.time()
lasso_perm = Lasso(alpha=best_alpha, max_iter=10000, random_state=21)
lasso_perm.fit(X_train_perm, y_train)
perm_time = time.time() - start_time
print(f"Время вычисления: {perm_time:.2f} сек")

df_temp = calculate_metrics(lasso_perm, X_train_perm, y_train, X_val_perm, y_val, X_test_perm, y_test, 'lasso_perm')
all_results = pd.concat([all_results, df_temp], ignore_index=True)
print(df_temp)

Время вычисления: 0.03 сек
        model  split          MAE          RMSE        R2
0  lasso_perm  train   959.254382   8983.339842  0.032057
1  lasso_perm    val  2031.925752   2714.915942 -0.027412
2  lasso_perm   test  2145.746987  46630.044010  0.001234


In [278]:
sample = X_val.iloc[:100]
exp = shap.Explainer(best_lasso, sample)
shap_values = exp(sample)

shap_imp = pd.DataFrame({
    'feature': X_train.columns,
    'importance': np.abs(shap_values.values).mean(axis=0)
}).sort_values('importance', ascending=False)

top10_shap = shap_imp.head(10)['feature'].tolist()
print(f"Топ-10 признаков: {top10_shap}\n\n")
shap_imp

Топ-10 признаков: ['bathrooms', 'bedrooms', 'Doorman', 'LaundryinUnit', 'LaundryinBuilding', 'NoFee', 'HardwoodFloors', 'DogsAllowed', 'HighSpeedInternet', 'Elevator']




,feature,importance
0,bathrooms,2033.406412
1,bedrooms,580.899612
6,Doorman,541.554491
12,LaundryinUnit,239.173001
9,LaundryinBuilding,155.541853
8,NoFee,117.452910
4,HardwoodFloors,88.681655
5,DogsAllowed,88.656054
16,HighSpeedInternet,59.771324
2,Elevator,53.388108


In [279]:
X_train_shap = X_train[top10_shap]
X_val_shap = X_val[top10_shap]
X_test_shap = X_test[top10_shap]

start_time = time.time()
lasso_shap = Lasso(alpha=best_alpha, max_iter=10000, random_state=42)
lasso_shap.fit(X_train_shap, y_train)
shap_time = time.time() - start_time
print(f"Время вычисления: {shap_time:.2f} сек")

df_temp = calculate_metrics(lasso_shap, X_train_shap, y_train, X_val_shap, y_val, X_test_shap, y_test, 'lasso_shap')
all_results = pd.concat([all_results, df_temp], ignore_index=True)
print(df_temp)

Время вычисления: 0.02 сек
        model  split          MAE          RMSE        R2
0  lasso_shap  train   946.258549   8980.857490  0.032592
1  lasso_shap    val  1981.930867   2670.879921  0.005647
2  lasso_shap   test  2109.179578  46628.857454  0.001285


In [280]:
all_results

,model,split,MAE,RMSE,R2
0,lasso,train,947.773239,8979.525492,0.032879
1,lasso,val,1969.391452,2660.601480,0.013286
2,lasso,test,2097.436003,46627.944083,0.001324
3,lasso_top10,train,948.297320,8980.694522,0.032627
4,lasso_top10,val,1977.672451,2668.067386,0.007740
5,lasso_top10,test,2106.725617,46629.438912,0.001260
6,lasso_corr,train,952.037483,8984.689171,0.031766
7,lasso_corr,val,1994.974778,2684.681828,-0.004656
8,lasso_corr,test,2127.033282,46633.164966,0.001100
9,lasso_perm,train,959.254382,8983.339842,0.032057


In [281]:
test_results = all_results[all_results['split'] == 'test'].drop('split', axis=1).reset_index(drop=True)

times_dict = {
    'lasso': lasso_time,
    'lasso_top10': 0,
    'lasso_corr': corr_time,
    'lasso_perm': perm_time,
    'lasso_shap': shap_time
}

test_results['Время (сек)'] = test_results['model'].map(times_dict)

comparison = test_results[['model', 'Время (сек)', 'MAE', 'RMSE', 'R2']]
comparison.columns = ['Метод', 'Время (сек)', 'Test MAE', 'Test RMSE', 'Test R2']
comparison

,Метод,Время (сек),Test MAE,Test RMSE,Test R2
0,lasso,0.031758,2097.436003,46627.944083,0.001324
1,lasso_top10,0.000000,2106.725617,46629.438912,0.001260
2,lasso_corr,0.014092,2127.033282,46633.164966,0.001100
3,lasso_perm,0.032417,2145.746987,46630.044010,0.001234
4,lasso_shap,0.023964,2109.179578,46628.857454,0.001285


Лучшая модель по метрикам и приемлемая по времени - Lasso с permutation importance

In [282]:
class MyGridSearch:
    
    def __init__(self, X_train, y_train, X_test, y_test, param_grid=None, cv=5):
        self.X_train = X_train
        self.y_train = y_train
        self.X_test = X_test
        self.y_test = y_test
        self.param_grid = param_grid if param_grid else {
            'alpha': [0.001, 0.01, 0.1, 0.5, 1, 10],
            'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
        }
        self.cv = cv
        
    def run(self):
        
        alphas = self.param_grid['alpha']
        l1_ratios = self.param_grid['l1_ratio']
        
        start = time.time()
        best_score = -np.inf
        best_params = None
        best_model = None
        
        for a in alphas:
            for l1 in l1_ratios:
                
                scores = []
                kf = KFold(n_splits=self.cv, shuffle=True, random_state=21)
                
                for train_idx, val_idx in kf.split(self.X_train):
                    X_tr, X_val = self.X_train[train_idx], self.X_train[val_idx]
                    y_tr, y_val = self.y_train[train_idx], self.y_train[val_idx]
                    
                    model = ElasticNet(alpha=a, l1_ratio=l1, max_iter=10000, random_state=21)
                    model.fit(X_tr, y_tr)
                    score = model.score(X_val, y_val)
                    scores.append(score)
                    
                mean_score = np.mean(scores)
                if mean_score > best_score:
                    best_score = mean_score
                    best_params = {'alpha': a, 'l1_ratio': l1}
                    best_model = ElasticNet(**best_params, max_iter=10000, random_state=21)
                    best_model.fit(self.X_train, self.y_train)
                
        self.time = time.time() - start
        self.best_params = best_params
        self.best_score = best_score
        self.best_model = best_model
        return self
    
    def result(self):
        y_pred = self.best_model.predict(self.X_test)
        
        return {
            'method': 'Grid Search',
            'best_params': self.best_params,
            'cv_r2': self.best_score,
            'time': self.time,
            'test_r2': r2_score(self.y_test, y_pred),
            'test_mae': mean_absolute_error(self.y_test, y_pred),
            'test_rmse': np.sqrt(mean_squared_error(self.y_test, y_pred))
        }

In [283]:
X_full = pd.concat([X_train, X_val])
y_full = pd.concat([y_train, y_val])

X_full = X_full.values
y_full = y_full.values
X_test_arr = X_test.values
y_test_arr = y_test.values

param_grid = {
    'alpha': [0.001, 0.01, 0.1, 0.5, 1, 10],
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

param_dist = {
    'alpha': (0.001, 10),
    'l1_ratio': (0, 1)
}

param_space = {
    'alpha': (1e-3, 1e2),
    'l1_ratio': (0, 1)
}

results = []

In [284]:
grid = MyGridSearch(X_full, y_full, X_test_arr, y_test_arr, param_grid=param_grid, cv=3).run()
results.append(grid.result())
print(f"   Лучшие параметры: {grid.best_params}")
print(f"   Test R2: {results[-1]['test_r2']:.4f}")
print(f"   Время: {grid.time:.2f} сек")

   Лучшие параметры: {'alpha': 0.001, 'l1_ratio': 0.9}
   Test R2: 0.0017
   Время: 7.72 сек


In [285]:
class MyRandomSearch:
    
    def __init__(self, X_train, y_train, X_test, y_test, param_dist=None, n_iter=50, cv=5):
        self.X_train = X_train
        self.y_train = y_train
        self.X_test = X_test
        self.y_test = y_test
        self.param_dist = param_dist if param_dist else {
            'alpha': (0.001, 10),
            'l1_ratio': (0, 1)
        }
        self.n_iter = n_iter
        self.cv = cv
        
    def run(self):
        np.random.seed(21)
        start = time.time()
        best_score = -np.inf
        best_params = None
        best_model = None
        
        for _ in range(self.n_iter):
            alpha = np.random.uniform(*self.param_dist['alpha'])
            l1_ratio = np.random.uniform(*self.param_dist['l1_ratio'])
            
            scores = []
            kf = KFold(n_splits=self.cv, shuffle=True, random_state=42)
            for train_idx, val_idx in kf.split(self.X_train):
                X_tr, X_val = self.X_train[train_idx], self.X_train[val_idx]
                y_tr, y_val = self.y_train[train_idx], self.y_train[val_idx]
                
                model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=10000, random_state=42)
                model.fit(X_tr, y_tr)
                score = model.score(X_val, y_val)
                scores.append(score)
            
            mean_score = np.mean(scores)
            if mean_score > best_score:
                best_score = mean_score
                best_params = {'alpha': alpha, 'l1_ratio': l1_ratio}
                best_model = ElasticNet(**best_params, max_iter=10000, random_state=42)
                best_model.fit(self.X_train, self.y_train)
        
        self.time = time.time() - start
        self.best_params = best_params
        self.best_score = best_score
        self.best_model = best_model
        return self
        
    def result(self):
        y_pred = self.best_model.predict(self.X_test)
        return {
            'method': 'Random Search',
            'best_params': self.best_params,
            'cv_r2': self.best_score,
            'time': self.time,
            'test_r2': r2_score(self.y_test, y_pred),
            'test_mae': mean_absolute_error(self.y_test, y_pred),
            'test_rmse': np.sqrt(mean_squared_error(self.y_test, y_pred))
        }

In [286]:
random = MyRandomSearch(X_full, y_full, X_test_arr, y_test_arr, param_dist=param_dist, n_iter=30, cv=3).run()
results.append(random.result())
print(f"   Лучшие параметры: {random.best_params}")
print(f"   Test R2: {results[-1]['test_r2']:.4f}")
print(f"   Время: {random.time:.2f} сек")

   Лучшие параметры: {'alpha': 0.6966399751713929, 'l1_ratio': 0.8674044839930883}
   Test R2: 0.0007
   Время: 2.32 сек


In [287]:
def objective(trial, X_train, y_train):
    alpha = trial.suggest_float("alpha", 1e-4, 10, log=True)
    l1_ratio = trial.suggest_float("l1_ratio", 0, 1)
    
    model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=10000, random_state=21)
    
    folds = k_fold(X_train, k=5)
    
    scores = []
    for train_idx, test_idx in folds:
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_val, y_val = X_train.iloc[test_idx], y_train.iloc[test_idx]
        model.fit(X_tr, y_tr)
        scores.append(model.score(X_val, y_val))
        
    return np.mean(scores)

In [288]:
start_time = time.time()
study = optuna.create_study(direction='maximize')
study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=30)
print(f"Время: {time.time() - start_time:.2f} сек")

[I 2026-05-04 21:46:51,976] A new study created in memory with name: no-name-c118255c-c524-45c3-9acb-f5980e431b99


[I 2026-05-04 21:46:52,194] Trial 0 finished with value: 0.11370958854689275 and parameters: {'alpha': 0.9273151879867402, 'l1_ratio': 0.8276157432271426}. Best is trial 0 with value: 0.11370958854689275.
[I 2026-05-04 21:46:54,390] Trial 1 finished with value: 0.3511066262334457 and parameters: {'alpha': 0.00014308342360626425, 'l1_ratio': 0.4500330023774054}. Best is trial 1 with value: 0.3511066262334457.
[I 2026-05-04 21:46:54,847] Trial 2 finished with value: 0.19980784238951546 and parameters: {'alpha': 0.06895017926983218, 'l1_ratio': 0.541782444706983}. Best is trial 1 with value: 0.3511066262334457.
[I 2026-05-04 21:46:55,189] Trial 3 finished with value: 0.12079384748267989 and parameters: {'alpha': 0.268797623996617, 'l1_ratio': 0.4831534908695142}. Best is trial 1 with value: 0.3511066262334457.
[I 2026-05-04 21:46:55,334] Trial 4 finished with value: 0.017621271138156813 and parameters: {'alpha': 8.73984588045143, 'l1_ratio': 0.5822760367694186}. Best is trial 1 with value

Время: 17.99 сек


In [289]:
best_model = ElasticNet(**study.best_params, max_iter=10000, random_state=21)
best_model.fit(X_full, y_full)
y_pred = best_model.predict(X_test)

print(f"Лучшие параметры: {study.best_params}")
print("Лучший скор:", study.best_value)
print(f"Test R2: {r2_score(y_test, y_pred):.4f}")

Лучшие параметры: {'alpha': 0.002183590746293726, 'l1_ratio': 0.9993192034949816}
Лучший скор: 0.35147966766353445
Test R2: 0.0017


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ElasticNet was fitted without feature names
  warnings.warn(


Было использовано три метода оптимизации гиперпараметров для ElasticNet: Grid Search, Random Search и Optuna, а также протестировали Optuna с разными схемами кросс-валидации. Grid Search простой и понятный, но медленный, так как перебирает все комбинации параметров. Random Search работает быстрее, но находит параметры чуть хуже. Optuna показала лучший баланс скорости и качества благодаря поиску, который учится на предыдущих попытках. 